# Re-ranking with Cross-Encoders: cheap recall, then expensive precision — built from scratch

First-stage retrieval (a bi-encoder, BM25, or hybrid) ranks passages by **independent encodings** — fast and precomputable, but coarse. The truly-best passage often lands in the top-k yet not at the top, because a bi-encoder can't model fine-grained query↔document token interaction. A **cross-encoder** fixes this: it feeds the query and one passage through the transformer **together** (`[CLS] query [SEP] passage`) and emits a single relevance score — far more accurate, but it costs one forward pass per pair, so you only run it on the top-k the bi-encoder already retrieved.

This notebook builds the two-stage pipeline on a corpus of look-alike "Helios-7 leadership" passages where the bi-encoder buries the true answer — and measures the re-ranking lift with **nDCG@k** and **MRR**, every number printed by a cell. We assert before we claim.

## Step 1 — set up: import the canonical re-ranking functions

Everything uses the verified functions from `reranking.py` — the same code the concept page shows. The bi-encoder is `all-MiniLM-L6-v2` (chapter 3's embedder); the cross-encoder is `cross-encoder/ms-marco-MiniLM-L-6-v2`. Both load on CPU; the banner prints which backend is active (real model vs deterministic fallback) so there's no ambiguity about what produced the numbers.

In [1]:
import numpy as np

from reranking import (
    CORPUS, QUERY, GOLD_INDEX, RETRIEVE_K, FINAL_K, NDCG_KS,
    BiEncoderRetriever, CrossEncoderReranker, retrieve_then_rerank,
    ndcg_at_k, dcg_at_k, reciprocal_rank,
)

print('numpy:', np.__version__)
try:
    import torch
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    # `device` is the detected accelerator; the demo models load on CPU on purpose (deterministic),
    # so the logits/ranks below come from CPU -- not from `device`.
    print('torch:', torch.__version__, '| detected device:', device, '| demo models on: cpu')
except ImportError:
    print('torch: not installed')

bi = BiEncoderRetriever(CORPUS)
cross = CrossEncoderReranker()
print('corpus passages:', len(CORPUS), '| retrieve_k =', RETRIEVE_K, '-> final_k =', FINAL_K)
print('bi-encoder backend :', bi.backend)
print('cross-encoder backend:', cross.backend)

numpy: 2.4.6


torch: 2.12.0 | detected device: mps | demo models on: cpu


corpus passages: 12 | retrieve_k = 10 -> final_k = 3
bi-encoder backend : sentence-transformers/all-MiniLM-L6-v2
cross-encoder backend: sentence-transformers/cross-encoder/ms-marco-MiniLM-L-6-v2


## Step 2 — the corpus: many look-alike leadership passages, one true answer

The query asks **who the deputy project lead is**. `doc[0]` answers it exactly. The rest are topically-near distractors — other Helios-7 roles and leadership statements — chosen so the bi-encoder, which scores by overall topical similarity, ranks several of them *above* the true answer.

In [2]:
print('QUERY:', QUERY)
print('GOLD :', f'doc[{GOLD_INDEX}]', CORPUS[GOLD_INDEX])
print()
for i, d in enumerate(CORPUS):
    print(f'doc[{i:>2}] {d}{"   <- GOLD" if i == GOLD_INDEX else ""}')

QUERY: Who is the deputy project lead for Helios-7?
GOLD : doc[0] The deputy project lead for Helios-7 is Dr. Amara Okoye, who runs the Nairobi office.

doc[ 0] The deputy project lead for Helios-7 is Dr. Amara Okoye, who runs the Nairobi office.   <- GOLD
doc[ 1] Dr. Liang Wei serves as the principal investigator of the Helios-7 science mission.
doc[ 2] Helios-7's mission operations are directed by flight director Captain Elena Reyes.
doc[ 3] The Helios-7 program is overseen by a steering committee of the partner agencies.
doc[ 4] The launch operations lead for Helios-7 coordinated the Kourou countdown sequence.
doc[ 5] Project leadership responsibilities for Helios-7 rotate annually among the agencies.
doc[ 6] The systems engineering lead for Helios-7 owns the spacecraft integration schedule.
doc[ 7] The Helios-7 ground segment lead manages the network of receiving stations.
doc[ 8] The chief scientist for the Helios-7 payload reports to the principal investigator.
doc[ 9] The deputy

## Step 3 — stage 1: the bi-encoder retrieves (and buries the gold)

The bi-encoder embeds the query and every passage independently, then ranks by cosine. Watch where the true answer lands: several vague leadership distractors outrank it, so the gold falls **outside the top-3** — if we fed only the top-3 to the generator, the right passage would be lost.

In [3]:
bi_full = bi.retrieve(QUERY, k=len(CORPUS))   # full bi-encoder ranking
bi_scores = bi.all_scores(QUERY)
print('bi-encoder ranking (doc ids, best-first):', list(bi_full.indices))
print()
for rank, doc in enumerate(bi_full.indices, 1):
    mark = '  <- GOLD' if doc == GOLD_INDEX else ''
    cutoff = '  | top-3 cutoff' if rank == FINAL_K else ''
    print(f'  {rank:>2}. doc[{doc:>2}] cos={bi_scores[doc]:.3f}{mark}{cutoff}')

bi_gold_rank = bi_full.indices.index(GOLD_INDEX) + 1
print(f'\ngold doc[{GOLD_INDEX}] bi-encoder rank: #{bi_gold_rank}  (outside the top-{FINAL_K})')
assert bi_gold_rank > FINAL_K, 'the bi-encoder should bury the gold past the top-k'

bi-encoder ranking (doc ids, best-first): [3, 5, 8, 0, 10, 6, 2, 1, 4, 9, 11, 7]

   1. doc[ 3] cos=0.744
   2. doc[ 5] cos=0.742
   3. doc[ 8] cos=0.730  | top-3 cutoff
   4. doc[ 0] cos=0.720  <- GOLD
   5. doc[10] cos=0.695
   6. doc[ 6] cos=0.639
   7. doc[ 2] cos=0.616
   8. doc[ 1] cos=0.612
   9. doc[ 4] cos=0.566
  10. doc[ 9] cos=0.549
  11. doc[11] cos=0.532
  12. doc[ 7] cos=0.478

gold doc[0] bi-encoder rank: #4  (outside the top-3)


## Step 4 — stage 2: the cross-encoder re-ranks the pool

Now feed each (query, passage) pair through the cross-encoder **together**. It emits a relevance logit per pair (an *uncalibrated* score — higher is more relevant, not a probability). Re-sorting the bi-encoder's top-k pool by this score lifts the true answer to **#1**.

In [4]:
pool = bi.retrieve(QUERY, k=RETRIEVE_K).indices          # the candidates we re-rank
ce_scores = cross.scores_for(QUERY, [CORPUS[i] for i in pool])
reranked = cross.rerank(QUERY, pool, CORPUS)

print(f're-ranked pool (top-{RETRIEVE_K}) by cross-encoder logit, best-first:')
for rank, (doc, s) in enumerate(zip(reranked.indices, reranked.scores), 1):
    mark = '  <- GOLD' if doc == GOLD_INDEX else ''
    print(f'  {rank:>2}. doc[{doc:>2}] logit={s:+.3f}{mark}')

rr_gold_rank = reranked.indices.index(GOLD_INDEX) + 1
print(f'\ngold doc[{GOLD_INDEX}] re-ranked rank: #{rr_gold_rank}')
assert rr_gold_rank == 1, 'the cross-encoder must lift the gold to #1'

re-ranked pool (top-10) by cross-encoder logit, best-first:
   1. doc[ 0] logit=+11.125  <- GOLD
   2. doc[ 9] logit=+6.459
   3. doc[ 1] logit=+4.427
   4. doc[ 6] logit=+3.907
   5. doc[ 2] logit=+3.683
   6. doc[10] logit=+2.531
   7. doc[ 5] logit=+2.505
   8. doc[ 8] logit=+1.942
   9. doc[ 4] logit=+1.624
  10. doc[ 3] logit=+1.144

gold doc[0] re-ranked rank: #1


![The re-rank shuffle: each candidate's bi-encoder rank (left) vs its cross-encoder re-ranked rank (right). The gold (green) climbs from #4, below the top-3 cutoff, to #1; distractors shuffle down. Generated by `make_figures_06.py`.](../../images/rag06_rank_shuffle.png)

## Step 5 — measure the lift: nDCG@k and MRR

We score ranking quality two ways, both from scratch:

- **nDCG@k** — Discounted Cumulative Gain, normalized: `DCG = Σ rel_i / log2(i+1)` over the top-k, divided by the ideal DCG (gold at rank 1). With one relevant doc, nDCG@k is 1.0 if the gold is #1, ~0.63 at #2, and **0.0 if the gold is past rank k**.
- **MRR** — Mean Reciprocal Rank: `1/rank` of the gold.

The bi-encoder ranks the gold #4, so **nDCG@3 = 0.0** (gold not in the top-3) and **MRR = 0.25**. After re-ranking, the gold is #1, so both hit **1.0**.

In [5]:
print(f"{'metric':<10} | {'bi-encoder':>11} | {'re-ranked':>10}")
print('-' * 38)
for k in NDCG_KS:
    before = ndcg_at_k(bi_full.indices, GOLD_INDEX, k)
    after = ndcg_at_k(reranked.indices, GOLD_INDEX, k)
    print(f"{'nDCG@'+str(k):<10} | {before:>11.3f} | {after:>10.3f}")
mrr_before = reciprocal_rank(bi_full.indices, GOLD_INDEX)
mrr_after = reciprocal_rank(reranked.indices, GOLD_INDEX)
print(f"{'MRR':<10} | {mrr_before:>11.3f} | {mrr_after:>10.3f}")

# Correctness BEFORE the claim
assert ndcg_at_k(bi_full.indices, GOLD_INDEX, FINAL_K) == 0.0, 'gold is past the top-3 for the bi-encoder'
assert ndcg_at_k(reranked.indices, GOLD_INDEX, FINAL_K) == 1.0, 're-ranked gold is #1'
assert mrr_after > mrr_before, 're-ranking must raise MRR'
print('\nre-ranking lifts nDCG@3 0.000 -> 1.000 and MRR 0.250 -> 1.000: True')

metric     |  bi-encoder |  re-ranked
--------------------------------------
nDCG@3     |       0.000 |      1.000
nDCG@5     |       0.431 |      1.000
nDCG@10    |       0.431 |      1.000
MRR        |       0.250 |      1.000

re-ranking lifts nDCG@3 0.000 -> 1.000 and MRR 0.250 -> 1.000: True


**Why nDCG discounts by rank — the numbers.** The log discount `1/log2(i+1)` is what makes ranking the answer *higher* score more. Here is the gain the gold contributes at each rank:

In [6]:
print('rank i | 1/log2(i+1)  (the gold\'s nDCG if placed at rank i)')
for i in range(1, 6):
    print(f'   {i}   |   {1/np.log2(i+1):.3f}')
# sanity: the bi-encoder put the gold at #4 -> its nDCG@5 is exactly this
print(f'\nbi-encoder gold at rank {bi_full.indices.index(GOLD_INDEX)+1}: nDCG@5 =',
      f'{ndcg_at_k(bi_full.indices, GOLD_INDEX, 5):.3f}  (= 1/log2(4+1))')

rank i | 1/log2(i+1)  (the gold's nDCG if placed at rank i)
   1   |   1.000
   2   |   0.631
   3   |   0.500
   4   |   0.431
   5   |   0.387

bi-encoder gold at rank 4: nDCG@5 = 0.431  (= 1/log2(4+1))


![Re-ranking quality lift — nDCG@k and MRR, bi-encoder (retrieve only) vs + cross-encoder re-rank. Every bar is the metric this notebook computes; re-ranking takes nDCG@3 from 0.00 to 1.00 and MRR from 0.25 to 1.00. Generated by `make_figures_06.py`.](../../images/rag06_ndcg_mrr.png)

## Step 6 — the recall ceiling: re-ranking reorders, it cannot rescue

A cross-encoder can only **reorder the candidates it is given**. If first-stage retrieval misses the gold, no re-ranker recovers it. We prove it: retrieve only the **top-3** (which excludes the gold), then re-rank — the gold is gone for good. The fix is to retrieve a **wide pool** (k=10 here) so the gold is present for re-ranking to find.

In [7]:
narrow_pool = bi.retrieve(QUERY, k=FINAL_K).indices       # retrieve ONLY top-3
narrow_rerank = cross.rerank(QUERY, narrow_pool, CORPUS).indices
print(f'retrieve top-{FINAL_K} -> pool {list(narrow_pool)} | gold doc[{GOLD_INDEX}] in pool:', GOLD_INDEX in narrow_pool)
if GOLD_INDEX in narrow_rerank:
    print(f'  re-ranked gold rank: #{narrow_rerank.index(GOLD_INDEX)+1}')
else:
    print('  re-ranked gold rank: MISS -- unrecoverable (re-ranking reorders, it cannot retrieve)')

wide_pool = bi.retrieve(QUERY, k=RETRIEVE_K).indices       # retrieve top-10
wide_rerank = cross.rerank(QUERY, wide_pool, CORPUS).indices
print(f'\nretrieve top-{RETRIEVE_K} -> pool {list(wide_pool)} | gold in pool:', GOLD_INDEX in wide_pool)
print(f'  re-ranked gold rank: #{wide_rerank.index(GOLD_INDEX)+1}  (recovered)')

assert GOLD_INDEX not in narrow_pool and GOLD_INDEX not in narrow_rerank, 'top-3 pool excludes the gold'
assert wide_rerank.index(GOLD_INDEX) == 0, 'top-10 pool lets re-ranking find the gold at #1'
print('\nrecall ceiling confirmed: a gold missing from the pool is unrecoverable by re-ranking')

retrieve top-3 -> pool [3, 5, 8] | gold doc[0] in pool: False
  re-ranked gold rank: MISS -- unrecoverable (re-ranking reorders, it cannot retrieve)

retrieve top-10 -> pool [3, 5, 8, 0, 10, 6, 2, 1, 4, 9] | gold in pool: True
  re-ranked gold rank: #1  (recovered)

recall ceiling confirmed: a gold missing from the pool is unrecoverable by re-ranking


![The recall ceiling: retrieve top-3 (gold missing from the pool — unrecoverable) vs retrieve top-10 (gold present — re-ranked to #1). Re-ranking REORDERS the pool; it cannot retrieve what stage 1 missed. Generated by `make_figures_06.py`.](../../images/rag06_recall_ceiling.png)

## Step 7 — the cost asymmetry, and the library one-liner

**Why not cross-encode everything?** Because the cross-encoder runs one transformer forward pass *per (query, passage) pair* at query time, and its score can't be precomputed (it depends on the query). Cost grows linearly with the pool, so you keep the re-rank pool small (typically 50–100) and never run it over the whole corpus.

```python
# The library one-liner — this is exactly what cross.scores_for(...) wraps:
from sentence_transformers import CrossEncoder
ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
scores = ce.predict([(query, passage) for passage in candidate_passages])  # one logit per pair
```

**Recap.** Two stages: cheap bi-encoder RECALL (retrieve a wide top-k), then expensive cross-encoder PRECISION (re-rank that pool by joint encoding). The bi-encoder can't model query↔passage interaction, so it buries the true answer; the cross-encoder reads them together and lifts it to #1 — measured here as nDCG@3 0.0→1.0, MRR 0.25→1.0. But re-ranking can only reorder what retrieval fetched (the recall ceiling), and it costs k forward passes (keep k small).

![Two-stage retrieval funnel: the 12-passage corpus narrows to a bi-encoder top-10 pool (cheap, precomputable), re-ranked down to the final top-3 (expensive, query-time only). Generated by `make_figures_06.py`.](../../images/rag06_funnel.png)